# BMG Labtech VANTAstar

The VANTAstar is a monochromator-based multi-mode microplate reader from BMG Labtech. PyLabRobot drives it over its native USB interface: **absorbance** (endpoint, kinetic and spectra), top **fluorescence intensity** (endpoint, kinetic, excitation and emission spectra) with automatic gain and focus, orbital and spiral **well scanning**, **shaking** and the **incubator**; **luminescence** is implemented but untested.

```{device-card} bmg-vantastar
```

```{warning}
This driver has been tested on **one VANTAstar (firmware 1.12)** only. Its **luminescence** read has not been tested at all; the spectra, kinetic reads and well scanning have not yet run on a reader from PyLabRobot. `setup()` logs a warning. Please report how it behaves on your reader.
```

## Properties

| | |
|---|---|
| Measurement modes | absorbance (endpoint, kinetic, spectrum), fluorescence intensity (top: endpoint, kinetic, excitation and emission spectra), luminescence (top, untested) |
| Well scanning | orbital, spiral |
| Environment | incubator (heating), shaking (orbital, double orbital, linear) |
| Interface | USB (VID 0x0483 / PID 0xA29A) |
| Not supported | bottom reads, pathlength correction, matrix well scanning |

## Setup

Create the reader with the `VANTAstar` factory and call `setup()`: it opens the USB connection, initializes the reader (homes the axes, about 10 s) and reads its firmware version and serial number.

In [ ]:
from pylabrobot.bmg_labtech import VANTAstar

reader = VANTAstar(name="vantastar")
await reader.setup()
reader.driver.configuration.firmware_version

## Loading a plate

Open the drawer, assign a plate to the reader's plate holder, then close the drawer. The plate's geometry is sent with every read.

In [ ]:
from pylabrobot.resources import cor_96_wellplate_360uL_Fb

await reader.open_drawer()
plate = cor_96_wellplate_360uL_Fb(name="plate")
reader.plate_holder.assign_child_resource(plate)
await reader.close_drawer()

## Absorbance

Measure the optical density at one to eight wavelengths (nm). The defaults (22 flashes, 0.5 s settling) are Voyager's "precise" preset. The optical density is computed as Voyager computes it; it is not pathlength-corrected. Voyager reports optical densities above 3.5 as 3.5; PyLabRobot returns the computed values.

In [ ]:
results = await reader.measure_absorbance([600])
results[0].data["A1"]

### Well scanning

Instead of the centre of each well, the reader can average over an orbit or a spiral of a given diameter (mm) around it. Every measurement takes a `well_scan`.

In [ ]:
from pylabrobot.bmg_labtech.vantastar.driver.master import WellScan

results = await reader.measure_absorbance([600], well_scan=WellScan("orbital", diameter=3))

### Absorbance spectrum

The reader's spectrometer reads every wavelength of a spectrum at once, well by well (220-1000 nm, in whole nm steps). The result holds each well's spectrum.

In [ ]:
spectrum = await reader.measure_absorbance_spectrum(220, 1000, step_size=1, wells=plate.get_items("A1:A3"))
spectrum.data["A1"][450]

### Kinetic reads

`measure_absorbance_kinetic` reads the plate `cycles` times, the reader starting a cycle every `cycle_time` seconds; it returns one list of results per cycle. `measure_fluorescence_kinetic` does the same for fluorescence.

In [ ]:
cycles = await reader.measure_absorbance_kinetic([450], cycles=6, cycle_time=10, wells=plate.get_items("A1:A3"))
[cycle[0].data["A1"] for cycle in cycles]

## Fluorescence intensity

Excitation and emission are set through the monochromator: a wavelength and a bandwidth (8-100 nm) each. The band edges are the nominal ones, wavelength +/- bandwidth / 2, and the dichroic is set midway between them, as Voyager sets it automatically.

Voyager places the band edges closer together, by an amount its filter software computes from the wavelength, the bandwidth and the slit. Narrow bands differ most: for excitation 365/8 nm and emission 400/10 nm Voyager uses 362.9-367.1 nm and 397.8-402.2 nm, where the nominal edges are 361-369 nm and 395-405 nm; bands of 20 nm and more differ by a few tenths of a nm. Readings at narrow bandwidths are therefore not directly comparable with Voyager's. To read with Voyager's edges, pass them as `monochromator`.

In [ ]:
result = await reader.measure_fluorescence(
  excitation_wavelength=365,
  emission_wavelength=400,
  excitation_bandwidth=8,
  emission_bandwidth=10,
  focal_height=5.0,
  gain=500,
)
result.data["A1"]

In [ ]:
from pylabrobot.bmg_labtech.vantastar.driver.master import MonochromatorSettings

voyager_365_400 = MonochromatorSettings(
  excitation_lower=362.9,
  excitation_upper=367.1,
  dichroic=382.0,
  emission_lower=397.8,
  emission_upper=402.2,
)
result = await reader.measure_fluorescence(
  excitation_wavelength=365,
  emission_wavelength=400,
  excitation_bandwidth=8,
  emission_bandwidth=10,
  focal_height=5.0,
  gain=500,
  monochromator=voyager_365_400,
)

### Automatic gain and focus

`adjust_gain_and_focus` lets the reader find the gain and the focal height on reference wells, as Voyager's automatic gain and focus does. Pass the result to `measure_fluorescence`. With `enhanced_dynamic_range=True` no gain is needed.

`adjust_focus=False` finds the gain alone at the given `focal_height`; `adjust_gain=False` finds the focal height alone. Voyager's autofocus runs both on the reference wells, then the focus alone on every well of the read, and reads at that height.

In [ ]:
reference_wells = plate.get_items("A1:H1")
adjusted = await reader.adjust_gain_and_focus(
  excitation_wavelength=365,
  emission_wavelength=400,
  excitation_bandwidth=8,
  emission_bandwidth=10,
  wells=reference_wells,
)
result = await reader.measure_fluorescence(
  excitation_wavelength=365,
  emission_wavelength=400,
  excitation_bandwidth=8,
  emission_bandwidth=10,
  focal_height=adjusted.focal_height,
  gain=adjusted.gain,
)

### Fluorescence spectra

`measure_fluorescence_spectrum` scans the excitation or the emission monochromator while the other stays fixed, with the gain and focal height given. The counts are the reader's raw counts: Voyager shows spectra multiplied by BMG's instrument correction curve, a factor per wavelength, so its exported spectra differ from these by that factor.

To measure at two excitation and emission pairs, read them one after another (Voyager reads them as separate passes too).

In [ ]:
emission = await reader.measure_fluorescence_spectrum(
  "emission",
  480,
  700,
  excitation_wavelength=450,
  excitation_bandwidth=16,
  emission_bandwidth=10,
  step_size=2,
  gain=1500,
  focal_height=6.8,
  wells=plate.get_items("H1:H3"),
)
emission.data["H1"][600]

## Luminescence

```{warning}
Luminescence is **untested**: neither the command nor the decoding of its data has run on a reader with luminescence. Please report how it behaves.
```

The defaults are Voyager's: all the emitted light, the reader's automatic gain, 0.8 s per well, 0.5 mm aperture distance.

In [ ]:
result = await reader.measure_luminescence(integration_time=0.8)
result.data["A1"]

## Shaking

Shake the plate orbital, double orbital or linear; the speed is in rpm, a multiple of 100, the duration in seconds.

In [ ]:
await reader.shake("double_orbital", speed=300, duration=10)

## Temperature

Heat the incubator, read its temperature, and switch heating off again (the temperature stays monitored).

In [ ]:
await reader.set_temperature(30.0)
await reader.get_temperature()

In [ ]:
await reader.stop_temperature_control()

## Teardown

Stop any running measurement, close the drawer and disconnect. `stop` refuses while a plate is still in the reader, unless you pass `accept_plate_left_in_device=True`.

In [ ]:
await reader.open_drawer()
reader.plate_holder.unassign_child_resource(plate)
await reader.close_drawer()
await reader.stop()